# SendKeyOnSound

Python port of `SendKeyOnSound.linq`.

The notebook polls the peak level of the default audio output. When the level exceeds a threshold it posts
`WM_KEYDOWN`/`WM_KEYUP` for a virtual key to every process with a given name, waits 2 s and sends the key again.
If nothing is heard for `MAX_ROUNDS_SECONDS`, the key is sent anyway as a keep-alive.

Windows only. Virtual key codes: https://learn.microsoft.com/en-us/windows/win32/inputdev/virtual-key-codes

In [27]:
# Run once
%pip install pycaw comtypes psutil

peak:    0.00Requirement already satisfied: pycaw in d:\Projects\GitHub\Nampacx\SendKeyOnSound\.venv\Lib\site-packages (20260927)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import ctypes
import threading
import time
from ctypes import POINTER, cast, wintypes

import comtypes
import psutil
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioMeterInformation

peak:    0.00

## Settings

Each setting has its own cell. Change a value and re-run only that cell (no restart needed, even while running).

In [29]:
# Key to send (virtual key code). 0x74 = F5
VIRTUALKEY = 0x74

In [30]:
# Target process name, case-insensitive, e.g. "notepad.exe". Empty matches nothing.
PROCESS_NAME = "wow.exe"

In [31]:
# Sound detected when peak * 1000 exceeds this
THRESHOLD = 150

In [33]:
POLL_SECONDS = 0.5          # how often the audio level is checked
MAX_ROUNDS_SECONDS = 22     # send a keep-alive key after this long without sound
REPEAT_DELAY_SECONDS = 1    # delay before the second key press after a sound event
HOTKEY_POLL_SECONDS = 0.05  # how often the hotkey (VIRTUALKEY) state is checked

In [34]:
user32 = ctypes.windll.user32
WM_KEYDOWN = 0x0100
WM_KEYUP = 0x0101

user32.PostMessageW.argtypes = [wintypes.HWND, wintypes.UINT, wintypes.WPARAM, wintypes.LPARAM]
user32.PostMessageW.restype = wintypes.BOOL
WNDENUMPROC = ctypes.WINFUNCTYPE(wintypes.BOOL, wintypes.HWND, wintypes.LPARAM)


def main_window_handles(pid):
    """First visible, titled top-level window of a process (like .NET's Process.MainWindowHandle)."""
    found = []

    def callback(hwnd, _):
        owner = wintypes.DWORD()
        user32.GetWindowThreadProcessId(hwnd, ctypes.byref(owner))
        if owner.value == pid and user32.IsWindowVisible(hwnd) and user32.GetWindowTextLengthW(hwnd) > 0:
            found.append(hwnd)
        return True

    user32.EnumWindows(WNDENUMPROC(callback), 0)
    return found[:1]


def list_applications():
    """Print every process that has a visible, titled window, to help pick PROCESS_NAME."""
    apps = {}

    def callback(hwnd, _):
        length = user32.GetWindowTextLengthW(hwnd)
        if user32.IsWindowVisible(hwnd) and length > 0:
            buf = ctypes.create_unicode_buffer(length + 1)
            user32.GetWindowTextW(hwnd, buf, length + 1)
            pid = wintypes.DWORD()
            user32.GetWindowThreadProcessId(hwnd, ctypes.byref(pid))
            apps.setdefault(pid.value, []).append(buf.value)
        return True

    user32.EnumWindows(WNDENUMPROC(callback), 0)
    rows = []
    for pid, titles in apps.items():
        try:
            rows.append((psutil.Process(pid).name(), pid, titles[0]))
        except psutil.Error:
            pass
    print("Running applications with a window:")
    for name, pid, title in sorted(rows, key=lambda r: r[0].lower()):
        print(f"  {name:<30} pid {pid:<7} {title}")


def find_targets():
    """Return [(pid, hwnd)] for the main window of every process matching PROCESS_NAME."""
    if not PROCESS_NAME:
        return []
    targets = []
    for proc in psutil.process_iter(["name"]):
        if (proc.info["name"] or "").lower() != PROCESS_NAME.lower():
            continue
        targets += [(proc.pid, hwnd) for hwnd in main_window_handles(proc.pid)]
    return targets


def click():
    """Post key down/up to the main window of every process matching PROCESS_NAME."""
    for _, hwnd in find_targets():
        user32.PostMessageW(hwnd, WM_KEYDOWN, VIRTUALKEY, 0)
        user32.PostMessageW(hwnd, WM_KEYUP, VIRTUALKEY, 0)


def open_meter():
    """Peak meter of the default render endpoint. Must be called on the thread that uses it."""
    speakers = AudioUtilities.GetSpeakers()
    speakers = getattr(speakers, "_dev", speakers)  # newer pycaw wraps the device
    interface = speakers.Activate(IAudioMeterInformation._iid_, CLSCTX_ALL, None)
    return cast(interface, POINTER(IAudioMeterInformation))

In [35]:
stop_event = threading.Event()
worker = None


def run():
    comtypes.CoInitialize()
    try:
        meter = open_meter()
        click()
        last_activity = time.monotonic()
        while not stop_event.wait(POLL_SECONDS):
            peak = meter.GetPeakValue() * 1000
            print(f"\rpeak: {peak:7.2f}", end="", flush=True)
            if peak > THRESHOLD:
                print("  sound detected")
                click()
                if stop_event.wait(REPEAT_DELAY_SECONDS):
                    break
                click()
                last_activity = time.monotonic()
            elif time.monotonic() - last_activity >= MAX_ROUNDS_SECONDS:
                click()
                last_activity = time.monotonic()
    finally:
        comtypes.CoUninitialize()
        print("\nstopped")


def start():
    global worker
    if worker and worker.is_alive():
        print("already running")
        return
    if not PROCESS_NAME:
        print("PROCESS_NAME is empty - set it first")
        return
    targets = find_targets()
    if not targets:
        print(f"no window found for {PROCESS_NAME!r} - is it running?")
        list_applications()
        return
    for pid, hwnd in targets:
        print(f"found {PROCESS_NAME} (pid {pid}, window {hwnd:#x})")
    stop_event.clear()
    worker = threading.Thread(target=run, daemon=True)
    worker.start()


def stop():
    stop_event.set()
    if worker:
        worker.join(timeout=5)

In [ ]:
hotkey_stop = threading.Event()
hotkey_worker = None


def key_is_down():
    return bool(user32.GetAsyncKeyState(VIRTUALKEY) & 0x8000)


def hotkey_loop():
    """Each press of VIRTUALKEY (in any window) runs stop() and start() together, i.e. a restart."""
    was_down = key_is_down()
    while not hotkey_stop.wait(HOTKEY_POLL_SECONDS):
        down = key_is_down()
        if down and not was_down:  # key-down transition only, not while held
            print("\nhotkey: restarting")
            stop()
            start()
            down = key_is_down()
        was_down = down


def listen():
    global hotkey_worker
    if hotkey_worker and hotkey_worker.is_alive():
        print("already listening")
        return
    hotkey_stop.clear()
    hotkey_worker = threading.Thread(target=hotkey_loop, daemon=True)
    hotkey_worker.start()
    print("listening: press the key to restart (stop + start)")


def unlisten():
    hotkey_stop.set()


## Run

`start()` runs the loop in a background thread, so the notebook stays usable. Call `stop()` to end it.
Set `PROCESS_NAME` above first, otherwise no key is ever sent.

Or run `listen()` once: from then on every press of the key (default F5, in any window) runs stop() and
start() together, restarting the whole sequence. Call `unlisten()` to turn that off.

In [36]:
# Optional: check that the target window is found before starting
targets = find_targets()
if not targets:
    list_applications()
targets

[(8508, 198022)]

In [40]:
start()

found wow.exe (pid 8508, window 0x30586)


peak:    0.00

In [39]:
stop()

In [ ]:
listen()